# ДЗ 3 — ver.3: временной тренд с замедлением роста

Итоговый алгоритм — **взвешенная LinearRegression логарифма цены**, как в ver.2. Деревья и бустинг не используются. Создаём `submission.csv` в папке ver.3.

Последнее изменение касается признака времени: сравниваем прежний кусочно-линейный тренд с замедляющимся ростом и выходом на плато. Форма тренда выбирается по локальному MAE, не по public-ответам.

Public соревнования равен $1/(1+MAE)$: большее значение Public лучше. Не смешиваем его с отдельно отображаемым полем «Баллы». Все локальные значения ниже — MAE в исходных единицах цены.

Ноутбук самодостаточен. Нужны три исходных CSV в `data/`, `../data/` или рядом с ноутбуком; зависимости указаны в requirements.txt.

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['OMP_NUM_THREADS'] = '2'
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error
from threadpoolctl import threadpool_limits
threadpool_limits(limits=2)
DATA_DIR = next((p for p in [Path('data'), Path('../data'), Path('.')]
                 if (p / 'train.csv').exists()), Path('data'))
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
X = train.drop(columns='price')
y = train.price
assert y.gt(0).all() and y.notna().all()
assert train.id.is_unique and test.id.is_unique
assert not set(train.id).intersection(test.id)
assert list(X.columns) == list(test.columns)
assert sample.id.is_unique and set(sample.id) == set(test.id)
dates = pd.to_datetime(train.date, format='%Y-%m')
print('Train:', train.shape, 'Test:', test.shape)

Train: (100000, 25) Test: (100000, 24)


## Признаки и модель

Сохраняем удачные признаки ver.2: One-Hot улиц, технологии здания и дискретных характеристик, сочетания технологии с площадью/комнатами, метро, лифтом и балконом. Числовая ветка: SimpleImputer и StandardScaler. Все обучаемые преобразования находятся внутри Pipeline и ColumnTransformer.

Веса для МНК: $a_i=\sqrt{y_i/\operatorname{median}(y_{train})}$. Они вычисляются только по обучающей части; при предсказании test значения цены не нужны.

Для нового признака времени проверяем

$$h(t)=q\frac{\min(t,18)}{18}+(1-q)\frac{\log(1+t/6)}{\log4},$$

где $t$ — число месяцев с января 2011, а $q$ выбирается из 0, 0.25, 0.5, 0.75, 1. При q=1 влияние времени после июля 2012 выходит на плато; при q=0 сохраняется замедляющийся логарифмический рост. Это преобразование входного признака: модель остаётся линейной по обучаемым коэффициентам.

Граница плато 18 месяцев выбрана в предварительном поиске среди 12, 15, 18, 21, 24; типы трендов также сравнивались предварительно. Поэтому локальные оценки относятся к повторно используемой валидации и могут быть оптимистичными.

In [2]:
"""Weighted ordinary linear regression for the real-estate homework.

The estimator is linear in the coefficients of log(price). No tree models are used.
All learned preprocessing statistics are fitted on the training fold only.
"""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted


class Version3Features(TransformerMixin, BaseEstimator):
    def __init__(self, trend='blend', mixing=1.0):
        self.trend = trend
        self.mixing = mixing

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # These transformations use no statistics and no target values.
        x = X.drop(columns=['id', 'price', 'date'], errors='ignore').copy().fillna(-1)
        date = pd.to_datetime(X['date'], format='%Y-%m')
        t = (date.dt.year - 2011) * 12 + date.dt.month - 1
        numeric = pd.DataFrame({
            'time': t, 'time2': np.maximum(t - 11, 0),
            'logarea': np.log(x.area), 'area': x.area,
            'arearoom': x.area / x.rooms,
        })
        if self.trend in ['linear', 'blend']:
            numeric = numeric.drop(columns='time2')
        if self.trend == 'blend':
            numeric['time'] = (self.mixing * np.minimum(t, 18) / 18
                               + (1 - self.mixing) * np.log1p(t / 6) / np.log(4))
        for c in [f'kw{i}' for i in range(1, 14)]:
            numeric[c] = x[c]
        categorical = x[['street_id', 'build_tech', 'g_lift', 'floor',
                         'rooms', 'metro_dist', 'balcon', 'n_photos']].astype(str)
        groups = [
            ['rooms', 'area'], ['floor', 'build_tech'], ['rooms', 'build_tech'],
            ['rooms', 'area', 'build_tech'], ['metro_dist', 'build_tech'],
            ['g_lift', 'build_tech'], ['balcon', 'build_tech'],
        ]
        for columns in groups:
            categorical['_'.join(columns)] = x[columns].astype(str).agg('_'.join, axis=1)
        return pd.concat([numeric, categorical], axis=1)


class Version3LinearRegressor(RegressorMixin, BaseEstimator):
    """OLS for log-price with training weights (price / median(price)) ** power."""
    def __init__(self, weight_power=0.5, trend='blend', mixing=1.0):
        self.weight_power = weight_power
        self.trend = trend
        self.mixing = mixing

    def fit(self, X, y):
        target = np.asarray(y, dtype=float)
        if not (np.isfinite(target).all() and (target > 0).all()):
            raise ValueError('Prices must be finite and positive.')
        # Inspect dtypes only. Imputer, scaler and OHE are fitted by the pipeline.
        example = Version3Features(self.trend, self.mixing).transform(X.iloc[:1])
        num_cols = example.select_dtypes(include='number').columns.tolist()
        cat_cols = example.select_dtypes(exclude='number').columns.tolist()
        numeric = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ])
        categorical = Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoder', OneHotEncoder(handle_unknown='ignore')),
        ])
        preprocess = ColumnTransformer([
            ('numeric', numeric, num_cols),
            ('categorical', categorical, cat_cols),
        ], sparse_threshold=1.0)
        self.pipeline_ = Pipeline([
            ('features', Version3Features(self.trend, self.mixing)),
            ('preprocess', preprocess),
            ('regressor', LinearRegression(tol=1e-6)),
        ])
        weights = (target / np.median(target)) ** self.weight_power
        self.pipeline_.fit(X, np.log(target), regressor__sample_weight=weights)
        self.n_features_in_ = X.shape[1]
        return self

    def predict(self, X):
        check_is_fitted(self, 'pipeline_')
        return np.exp(self.pipeline_.predict(X))


## Проверка на двух временных разбиениях

- Train до октября 2011 → validation октябрь–декабрь 2011.
- Train до октября 2012 → validation октябрь–декабрь 2012.

Во второй проверке объединены все три поздних месяца для оценки более длинного горизонта; поэтому её числа нельзя напрямую сравнивать с таблицей ver.2, где для выбора использовались только октябрь–ноябрь. Ниже ver.2 переобучается и оценивается на **тех же** разбиениях, что и ver.3.

Свежего независимого holdout больше нет: все эти данные уже участвовали в анализе. Финальная внешняя оценка будет на соревновании. Параметры выбираем по среднему MAE двух периодов.

In [3]:
folds = [('2011 Q4', '2011-10-01', '2012-01-01'),
         ('2012 Q4', '2012-10-01', '2013-01-01')]
configs = {'ver.2': dict(trend='piecewise', weight_power=0.5)}
configs.update({f'q={q}': dict(trend='blend', mixing=q, weight_power=0.5)
                for q in [0., 0.25, 0.5, 0.75, 1.]})
rows = []
for fold, start, end in folds:
    fit = dates < start
    valid = (dates >= start) & (dates < end)
    assert dates[fit].max() < dates[valid].min()
    for name, config in configs.items():
        model = Version3LinearRegressor(**config)
        model.fit(X.loc[fit], y.loc[fit])
        p = model.predict(X.loc[valid])
        assert np.isfinite(p).all() and (p > 0).all()
        mae = mean_absolute_error(y.loc[valid], p)
        rows.append({'fold': fold, 'model': name, 'MAE': mae})
        print(fold, name, round(mae))
results = pd.DataFrame(rows)
table = results.pivot(index='model', columns='fold', values='MAE')
table['mean_MAE'] = table.mean(axis=1)
table = table.sort_values('mean_MAE')
display(table)
best_name = table.drop(index='ver.2').index[0]
best_config = configs[best_name].copy()
print('Выбрано:', best_config)
improvement = 1 - table.loc[best_name, 'mean_MAE'] / table.loc['ver.2', 'mean_MAE']
print(f'Изменение среднего MAE к ver.2: {-improvement:.2%}')
assert improvement > 0
results.to_csv('validation_results.csv', index=False)
metrics = {'best_config': best_config, 'cv_mae_v2': float(table.loc['ver.2', 'mean_MAE']),
           'cv_mae_v3': float(table.loc[best_name, 'mean_MAE']),
           'note': 'Repeated temporal validation; no fresh independent holdout.'}
Path('metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

2011 Q4 ver.2 605433


2011 Q4 q=0.0 629046


2011 Q4 q=0.25 623189


2011 Q4 q=0.5 617147


2011 Q4 q=0.75 611112


...

2012 Q4 q=0.75 632631


2012 Q4 q=1.0 636182


fold,2011 Q4,2012 Q4,mean_MAE
model,,,
q=1.0,605433.516681,636182.487707,620808.002194
q=0.75,611112.414669,632630.771395,621871.593032
q=0.5,617147.184023,629834.968579,623491.076301
q=0.25,623189.452858,627992.522757,625590.987808
q=0.0,629046.225215,627395.165742,628220.695478
ver.2,605432.700632,686792.794282,646112.747457


Выбрано: {'trend': 'blend', 'mixing': 1.0, 'weight_power': 0.5}
Изменение среднего MAE к ver.2: -3.92%


232

## Финальное обучение и файл для соревнования

Обучаем выбранный вариант на всех размеченных строках. Сохраняем ровно две колонки, id в порядке sample_submission. Проверяем корректность выгруженного CSV.

In [4]:
final_model = Version3LinearRegressor(**best_config).fit(X, y)
prediction = final_model.predict(test)
assert np.isfinite(prediction).all() and (prediction > 0).all()
submission = sample[['id']].copy()
submission['price'] = submission.id.map(pd.Series(prediction, index=test.id))
assert submission.shape == sample.shape and submission.id.equals(sample.id)
assert submission.notna().all().all()
submission.to_csv('submission.csv', index=False)
check = pd.read_csv('submission.csv')
assert check.id.equals(sample.id) and np.allclose(check.price, submission.price)
print('Сохранён submission.csv:', submission.shape)
display(submission.head())
display(submission.price.describe())
probe = test.iloc[:2].copy()
probe['street_id'] = -999
probe[['build_tech', 'g_lift', 'metro_dist']] = np.nan
assert np.isfinite(final_model.predict(probe)).all()
print('Неизвестная улица и пропуски обрабатываются корректно.')

Сохранён submission.csv: (100000, 2)


,id,price
0,100000,8.295154e+06
1,100001,4.761674e+06
2,100002,3.925155e+06
3,100003,3.118511e+06
4,100004,6.720483e+06


count    1.000000e+05
mean     5.777259e+06
std      4.550561e+06
min      5.752651e+05
25%      3.124999e+06
50%      4.598943e+06
75%      6.906081e+06
max      9.457277e+07
Name: price, dtype: float64

Неизвестная улица и пропуски обрабатываются корректно.


## Ограничения

Выигрыш небольшой и измерен на повторно используемой валидации. Public/private для ver.3 ещё неизвестны. Вариант с плато предполагает, что вклад общего времени перестаёт расти; это может оказаться неверным для 2013 года. Сохранены ver.1 и ver.2 для сравнения. Не считаем локальное улучшение гарантией улучшения на соревновании.